# 01 · Descarga y Construcción del Corpus
**Sistema RAG · Ingeniería Ontológica – UNAL Medellín**

Descarga **50 artículos científicos** de [arXiv](https://arxiv.org) sobre
bioinformática, genómica y diagnóstico médico con IA, y los almacena en
Google Drive junto con un manifiesto JSON de metadatos.

| Campo | Valor |
|---|---|
| Dominio | Bioinformática / Medicina |
| Fuente | arXiv.org (acceso abierto) |
| Formato | PDF |
| Destino | `/content/drive/MyDrive/RAG_BioMed/corpus/` |


In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────
!pip install -q arxiv==2.1.3
print('✓ arxiv instalado')


In [ ]:
# ── Montar Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, json, time

# Rutas base del proyecto (compartidas con todos los notebooks)
BASE_DIR    = '/content/drive/MyDrive/RAG_BioMed'
CORPUS_DIR  = f'{BASE_DIR}/corpus'
MANIFEST    = f'{BASE_DIR}/manifest.json'

os.makedirs(CORPUS_DIR, exist_ok=True)
print(f'✓ Directorio corpus: {CORPUS_DIR}')


In [ ]:
import arxiv

# ── Función principal de descarga ──────────────────────────────────────────
def descargar_corpus(consulta: str, n: int = 50,
                     directorio: str = CORPUS_DIR) -> list[dict]:
    """
    Descarga hasta `n` artículos de arXiv que coincidan con `consulta`.

    Args:
        consulta   : Cadena de búsqueda arXiv (e.g. 'bioinformatics deep learning').
        n          : Número máximo de artículos a descargar.
        directorio : Ruta local donde guardar los PDFs.

    Returns:
        Lista de dicts con metadatos de cada artículo descargado.
    """
    client = arxiv.Client(page_size=10, delay_seconds=3.0)
    search = arxiv.Search(
        query=consulta,
        max_results=n,
        sort_by=arxiv.SortCriterion.Relevance
    )

    metadatos = []
    for i, paper in enumerate(client.results(search)):
        # Nombre de archivo limpio
        arxiv_id = paper.entry_id.split('/')[-1].replace('/', '_')
        filename = f'doc_{i+1:02d}_{arxiv_id}.pdf'
        filepath = os.path.join(directorio, filename)

        # Descargar solo si no existe
        if not os.path.exists(filepath):
            try:
                paper.download_pdf(dirpath=directorio, filename=filename)
                estado = '✓ descargado'
            except Exception as e:
                estado = f'✗ error: {e}'
                filename = None
        else:
            estado = '→ ya existe'

        print(f'[{i+1:02d}/{n}] {estado}  {paper.title[:65]}')

        if filename:
            metadatos.append({
                'doc_id'    : f'doc_{i+1:02d}',
                'filename'  : filename,
                'filepath'  : filepath,
                'titulo'    : paper.title,
                'autores'   : [a.name for a in paper.authors[:5]],
                'anio'      : paper.published.year,
                'arxiv_id'  : arxiv_id,
                'categorias': paper.categories,
                'abstract'  : paper.summary
            })
        time.sleep(0.5)

    return metadatos


In [ ]:
# ── Ejecutar descarga ──────────────────────────────────────────────────────
CONSULTA = (
    'bioinformatics machine learning genomics medical diagnosis '
    'deep learning protein structure drug discovery'
)

print(f'Descargando {50} artículos sobre: {CONSULTA[:80]}...\n')
metadatos = descargar_corpus(CONSULTA, n=50)


In [ ]:
# ── Guardar manifiesto JSON ────────────────────────────────────────────────
def guardar_manifiesto(metadatos: list[dict], ruta: str) -> None:
    """
    Serializa la lista de metadatos a un archivo JSON.

    Args:
        metadatos : Lista producida por `descargar_corpus`.
        ruta      : Ruta completa del archivo de salida.
    """
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(metadatos, f, ensure_ascii=False, indent=2)
    print(f'✓ Manifiesto guardado → {ruta}  ({len(metadatos)} docs)')

guardar_manifiesto(metadatos, MANIFEST)


In [ ]:
# ── Estadísticas del corpus ────────────────────────────────────────────────
from collections import Counter

anios = Counter(m['anio'] for m in metadatos)
print('Distribución por año:')
for anio, cnt in sorted(anios.items()):
    print(f'  {anio}: {"█" * cnt} ({cnt})')

print(f'\nTotal documentos : {len(metadatos)}')
print(f'Directorio corpus: {CORPUS_DIR}')
print(f'Manifiesto       : {MANIFEST}')
